# [Final] EXAONE-4.0-1.2B AWQ-Style Quantization

이 노트북은 **llmcompressor**를 사용하여 **AWQ 스타일**의 양자화를 수행하는 최종 버전입니다.

**주요 정보:**
- **라이브러리**: `llmcompressor` (AutoAWQ 사용 안 함)
- **모델 로딩**: `AutoModelForCausalLM` (표준 방식)
- **양자화 방식**: `GPTQModifier`를 사용하여 AWQ와 동일한 설정 적용 (W4A16, Group=128)
- **호환성**: vLLM 및 평가 서버 환경과 100% 호환

# 1. Installation

In [ ]:
# 필수 라이브러리 설치
!pip install -q compressed-tensors>=0.13.0
!pip install -q llmcompressor>=0.13.0
!pip install -q transformers>=4.57.3
!pip install -q datasets
!pip install -q sentencepiece
!pip install -q accelerate

print("✓ Installation complete!")

# 2. Import

In [ ]:
import os
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
# AWQ는 GPTQ의 특수한 형태이므로 GPTQModifier를 사용합니다 (llmcompressor 방식)
from llmcompressor.modifiers.quantization import GPTQModifier

# 3. Setting

In [ ]:
MODEL_ID = "LGAI-EXAONE/EXAONE-4.0-1.2B"  # Hugging Face ID
OUT_DIR  = "./model_awq"

DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

NUM_CALIBRATION_SAMPLES = 256
MAX_SEQUENCE_LENGTH = 512

# AWQ 스타일 설정 (W4A16 Scheme)
SCHEME = "W4A16"
TARGETS = ["Linear"]
IGNORE  = ["embed_tokens", "lm_head"]

# 4. Model Loads

In [ ]:
print("[INFO] 모델 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

# 표준 AutoModelForCausalLM 사용
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

print("[INFO] 모델/토크나이저 로드 완료")

# 5. Dataset Loads & Preprocess

In [ ]:
print("[INFO] 캘리브레이션 데이터 로드 중...")

ds = load_dataset(
    DATASET_ID,
    split=f"{DATASET_SPLIT}[:{NUM_CALIBRATION_SAMPLES}]",
)

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False)
    }

ds = ds.map(preprocess)

print("[INFO] 데이터 전처리 완료")

# 6. AWQ-Style Quantization

In [ ]:
print(f"[INFO] AWQ 스타일 양자화 시작 (scheme={SCHEME})...")

# AWQ와 유사한 효과를 내기 위한 GPTQ 설정
# - scheme="W4A16": 4-bit weights, 16-bit activations
# - block_size=128: AWQ의 그룹 크기와 동일
# - actorder=False: 정적 순서 사용 (AWQ는 보통 static)
recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
        block_size=128,
        actorder="static",  
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("[INFO] 양자화 완료")

# 7. Model Save

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)

# save_compressed=True: 압축된 가중치 저장
model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

print(f"[INFO] 모델 저장 완료: {OUT_DIR}")

# 8. Submission

In [ ]:
zip_name = "awq_style_submit"
print(f"[INFO] {zip_name}.zip 생성 중...")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

print(f"[INFO] 생성 완료: {zip_name}.zip")